# Phase 3 | A/B Test Analysis

**Project**: Beyond Risk Scores: Uplift-Driven Financial Intervention for Loan Default Prevention

The matched cohorts from Phase 2 are balanced on all observable features. This notebook
simulates an intervention effect, tests whether it is statistically and practically
significant, validates the result with post-experiment power analysis, and models a
staged rollout from 5% canary deployment to full scale.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
from statsmodels.stats.power import NormalIndPower

In [3]:
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

COLORS = ['steelblue', 'cadetblue', 'skyblue', 'lightblue']

## Step 1 | Load Matched Cohorts

The matched cohorts from Phase 2 contain 35,952 treatment and 35,952 control customers,
balanced on EXT_SOURCE_2, EXT_SOURCE_3, DAYS_BIRTH, and DEBT_TO_INCOME with all SMD
values below 0.01.

In [5]:
df = pd.read_csv('../data/processed/matched_cohorts.csv')

treatment = df[df['GROUP'] == 'treatment'].copy()
control = df[df['GROUP'] == 'control'].copy()

print("=" * 45)
print("Matched Cohorts Loaded")
print("=" * 45)
print(f"Total customers        : {len(df):,}")
print(f"Treatment              : {len(treatment):,}")
print(f"Control                : {len(control):,}")
print(f"Columns                : {df.shape[1]}")
print()
print(f"Treatment default rate : {treatment['TARGET'].mean()*100:.2f}%")
print(f"Control default rate   : {control['TARGET'].mean()*100:.2f}%")
print(f"Difference             : {(treatment['TARGET'].mean() - control['TARGET'].mean())*100:.2f} pp")
print("=" * 45)

Matched Cohorts Loaded
Total customers        : 71,904
Treatment              : 35,952
Control                : 35,952
Columns                : 89

Treatment default rate : 9.59%
Control default rate   : 9.78%
Difference             : -0.19 pp


## Step 2 | Simulate Intervention Effect

The matched cohorts currently show no meaningful difference because no intervention has
occurred. In a real bank, the treatment group would have received collections calls and
some customers would have recovered as a result.

We simulate this using a stochastic approach. Each treatment customer who defaulted gets
a personal recovery probability based on their financial profile. Customers with stronger
external scores and lower debt burden are more likely to respond to intervention. This
reflects reality: a collections call works better on someone who is stressed but capable
than on someone who is financially overwhelmed.

Only treatment customers are modified. Control customers remain untouched. This preserves
the experimental structure.

In [6]:
np.random.seed(42)

# Only defaulted treatment customers can be "saved" by intervention
defaulted_treatment = treatment[treatment['TARGET'] == 1].index

# Recovery probability based on customer profile
# Higher EXT_SOURCE = stronger financial standing = more likely to respond
# Lower DEBT_TO_INCOME = more capacity to recover
def recovery_probability(row):
    ext2_score = row['EXT_SOURCE_2'] if not np.isnan(row['EXT_SOURCE_2']) else 0
    ext3_score = row['EXT_SOURCE_3'] if not np.isnan(row['EXT_SOURCE_3']) else 0
    dti = row['DEBT_TO_INCOME']
    
    # Combine signals: higher external scores and lower DTI = higher recovery chance
    prob = (ext2_score * 0.35) + (ext3_score * 0.35) + ((1 - min(dti, 1)) * 0.30)
    
    # Scale to realistic range: 15% to 40% recovery probability
    prob = 0.15 + (prob * 0.25)
    return prob

# Calculate personal recovery probability for each defaulted treatment customer
recovery_probs = treatment.loc[defaulted_treatment].apply(recovery_probability, axis=1)

# Simulate: each defaulted treatment customer rolls against their probability
recovered = np.random.random(len(defaulted_treatment)) < recovery_probs

# Flip recovered customers from default (1) to non-default (0)
treatment.loc[defaulted_treatment[recovered], 'TARGET'] = 0

new_treatment_rate = treatment['TARGET'].mean()
control_rate = control['TARGET'].mean()
observed_effect = control_rate - new_treatment_rate

print("=" * 45)
print("Intervention Simulation")
print("=" * 45)
print(f"Defaulted in treatment : {len(defaulted_treatment):,}")
print(f"Recovered via call     : {recovered.sum():,}")
print(f"Recovery rate          : {recovered.mean()*100:.2f}%")
print()
print(f"Treatment default rate : {new_treatment_rate*100:.2f}% (after intervention)")
print(f"Control default rate   : {control_rate*100:.2f}% (no intervention)")
print(f"Observed effect        : {observed_effect*100:.2f} pp reduction")
print("=" * 45)

Intervention Simulation
Defaulted in treatment : 3,447
Recovered via call     : 960
Recovery rate          : 27.85%

Treatment default rate : 6.92% (after intervention)
Control default rate   : 9.78% (no intervention)
Observed effect        : 2.86 pp reduction


### Intervention Simulation Finding

The stochastic simulation produced a 27.85% recovery rate among defaulted treatment customers.
Each customer's recovery probability was driven by their EXT_SOURCE scores and debt-to-income
ratio, so the intervention naturally helped stronger-profile customers more than weaker ones.

The observed effect is a 2.86 pp reduction in default rate (from 9.78% to 6.92%). This exceeds
our MDE of 2.0 pp. The next steps test whether this difference is statistically significant,
practically meaningful, and robust enough to trust.

## Step 3 | Statistical Significance Test

A 2.86 pp difference looks meaningful. But with 35,952 customers per group, could this
difference have occurred by chance? We test this with a two-proportion z-test.

- Null hypothesis: the default rate is the same in both groups
- Alternative hypothesis: the default rates are different
- Rejection threshold: alpha = 0.05

In [8]:
# Treatment: number of defaults and total customers
treatment_defaults = treatment['TARGET'].sum()
treatment_n = len(treatment)

# Control: number of defaults and total customers
control_defaults = control['TARGET'].sum()
control_n = len(control)

# Two-proportion z-test
count = np.array([treatment_defaults, control_defaults])
nobs = np.array([treatment_n, control_n])

z_stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')

# Confidence interval for the difference in proportions
diff = control['TARGET'].mean() - treatment['TARGET'].mean()
se = np.sqrt(
    (treatment['TARGET'].mean() * (1 - treatment['TARGET'].mean()) / treatment_n) +
    (control['TARGET'].mean() * (1 - control['TARGET'].mean()) / control_n)
)
ci_lower = diff - 1.96 * se
ci_upper = diff + 1.96 * se

print("=" * 45)
print("Two-Proportion Z-Test")
print("=" * 45)
print(f"Treatment defaults     : {treatment_defaults:,} / {treatment_n:,}")
print(f"Control defaults       : {control_defaults:,} / {control_n:,}")
print()
print(f"Treatment default rate : {treatment['TARGET'].mean()*100:.2f}%")
print(f"Control default rate   : {control['TARGET'].mean()*100:.2f}%")
print(f"Observed difference    : {diff*100:.2f} pp")
print()
print(f"Z-statistic            : {z_stat:.4f}")
print(f"P-value                : {p_value:.6f}")
print(f"95% CI for difference  : [{ci_lower*100:.2f} pp, {ci_upper*100:.2f} pp]")
print()
if p_value < 0.05:
    print("RESULT: Statistically significant at alpha = 0.05")
else:
    print("RESULT: Not statistically significant at alpha = 0.05")
print("=" * 45)

Two-Proportion Z-Test
Treatment defaults     : 2,487 / 35,952
Control defaults       : 3,515 / 35,952

Treatment default rate : 6.92%
Control default rate   : 9.78%
Observed difference    : 2.86 pp

Z-statistic            : -13.8603
P-value                : 0.000000
95% CI for difference  : [2.46 pp, 3.26 pp]

RESULT: Statistically significant at alpha = 0.05


### Statistical Significance Finding

The p-value is effectively zero. We reject the null hypothesis that the two groups have
the same default rate.

More importantly, the 95% confidence interval for the difference is [2.46 pp, 3.26 pp].
The lower bound of 2.46 pp sits above our MDE of 2.0 pp. This means even the most
conservative estimate of the intervention effect exceeds the minimum threshold we set
for operational relevance. The result is not just statistically significant. It is
significant at a level that matters.

## Step 5 | Practical Significance

A statistically significant result does not automatically mean a useful one. A p-value
can be tiny even when the effect is too small to act on. Practical significance asks:
does this effect clear the financial break-even threshold established in Phase 2, and
how much capital does it actually recover?

In [9]:
# Phase 2 economic parameters
cost_per_call = 50
lgd = 0.45
avg_loan = df['AMT_CREDIT'].mean()
recovery_value = avg_loan * lgd
break_even_uplift = cost_per_call / recovery_value

# Observed results
observed_effect = diff
defaults_prevented = recovered.sum()
total_treated = len(treatment)

# Financial impact
capital_recovered = defaults_prevented * recovery_value
total_intervention_cost = total_treated * cost_per_call
net_value = capital_recovered - total_intervention_cost
roi = (net_value / total_intervention_cost) * 100

print("=" * 45)
print("Practical Significance")
print("=" * 45)
print()
print("THRESHOLD CHECK")
print(f"  Break-even uplift (U_min) : {break_even_uplift*100:.4f}%")
print(f"  Observed effect           : {observed_effect*100:.2f}%")
print(f"  Clears break-even         : {'YES' if observed_effect > break_even_uplift else 'NO'}")
print(f"  Clears MDE (2.0 pp)       : {'YES' if observed_effect > 0.02 else 'NO'}")
print()
print("FINANCIAL IMPACT")
print(f"  Defaults prevented        : {defaults_prevented:,}")
print(f"  Loss avoided per save     : {recovery_value:,.0f} CU")
print(f"  Total capital recovered   : {capital_recovered:,.0f} CU")
print(f"  Total intervention cost   : {total_intervention_cost:,.0f} CU")
print(f"  Net value                 : {net_value:,.0f} CU")
print(f"  ROI                       : {roi:,.1f}%")
print("=" * 45)

Practical Significance

THRESHOLD CHECK
  Break-even uplift (U_min) : 0.0145%
  Observed effect           : 2.86%
  Clears break-even         : YES
  Clears MDE (2.0 pp)       : YES

FINANCIAL IMPACT
  Defaults prevented        : 960
  Loss avoided per save     : 346,004 CU
  Total capital recovered   : 332,163,365 CU
  Total intervention cost   : 1,797,600 CU
  Net value                 : 330,365,765 CU
  ROI                       : 18,378.2%


### Practical Significance Finding

The observed effect of 2.86 pp clears both thresholds. It is 197x larger than the financial
break-even uplift of 0.0145% and exceeds the 2.0 pp MDE set in Phase 2.

The intervention prevented 960 defaults, recovering 332.2M CU in capital that would have
been lost. The total cost of calling 35,952 treatment customers was 1.8M CU. Net value
is 330.4M CU with an ROI of 18,378%.

The result is not just statistically significant. It is financially significant at a scale
that justifies immediate action.

## Step 6 | Post-Experiment Power Check

In Phase 2, we calculated power before the experiment to size it correctly. Now we calculate
power after the experiment to confirm the result is robust.

A statistically significant result with low power is fragile. It could be a lucky sample.
A significant result with high power is reliable. This step answers: given the effect we
observed and the sample we used, how confident are we that this result would replicate?

In [10]:
# Observed effect size (Cohen's h)
observed_h = proportion_effectsize(
    control['TARGET'].mean(),
    treatment['TARGET'].mean()
)

# Post-experiment power calculation
post_analysis = NormalIndPower()
achieved_power = post_analysis.solve_power(
    effect_size=abs(observed_h),
    nobs1=len(treatment),
    alpha=0.05,
    alternative='two-sided'
)

print("=" * 45)
print("Post-Experiment Power Check")
print("=" * 45)
print(f"Observed effect (Cohen h) : {abs(observed_h):.4f}")
print(f"Sample per group          : {len(treatment):,}")
print(f"Alpha                     : 0.05")
print(f"Achieved power            : {achieved_power:.4f}")
print()
if achieved_power >= 0.80:
    print(f"VERDICT: Robust result. Power = {achieved_power:.2f}")
    print("The probability of detecting this effect is near certain.")
    print("This result would replicate reliably.")
else:
    print(f"VERDICT: Fragile result. Power = {achieved_power:.2f}")
    print("The result may not replicate. Larger sample needed.")
print("=" * 45)

Post-Experiment Power Check
Observed effect (Cohen h) : 0.1037
Sample per group          : 35,952
Alpha                     : 0.05
Achieved power            : 1.0000

VERDICT: Robust result. Power = 1.00
The probability of detecting this effect is near certain.
This result would replicate reliably.


### Post-Experiment Power Finding

Achieved power is 1.00. Given the observed effect size and sample used, this result would
replicate every time. Compare this to the Phase 2 requirement of 0.80: we exceeded it
completely.

This confirms two things. First, the Phase 2 power analysis correctly sized the experiment.
We needed 2,738 per group and used 35,952. Second, the observed 2.86 pp effect is not a
statistical fluke. It is a reliable, reproducible finding backed by overwhelming power.

## Step 7 | Controlled Rollout Modeling

A proven intervention effect does not mean immediate full deployment. Responsible rollout
happens in stages, each with a specific purpose:

- 5% Canary: Check for Sleeping Dogs. Are there customer segments where the intervention
  makes things worse? If yes, stop before scaling.
- 20% Validation: Confirm the uplift holds at a larger scale with statistical significance.
- 50% Scale: Half the at-risk population is now being called. ROI at this stage drives the
  budget request for full deployment.
- 100% Full Rollout: All at-risk customers receive the intervention. Final ROI calculation.

Each stage is modeled with its own sample size, expected defaults prevented, and net value.

In [11]:
rollout_stages = [0.05, 0.20, 0.50, 1.00]
stage_labels = ['5% Canary', '20% Validation', '50% Scale', '100% Full Rollout']

# Observed metrics from the experiment
observed_reduction = diff
intervention_recovery_rate = recovered.sum() / len(defaulted_treatment)

print("=" * 60)
print("Controlled Rollout Model")
print("=" * 60)
print(f"At-risk population     : {total_treated:,}")
print(f"Baseline default rate  : {control['TARGET'].mean()*100:.2f}%")
print(f"Observed reduction     : {observed_reduction*100:.2f} pp")
print(f"Recovery rate per call : {intervention_recovery_rate*100:.2f}%")
print()

rollout_results = []

for stage, label in zip(rollout_stages, stage_labels):
    n_customers = int(total_treated * stage)
    n_expected_defaults = int(n_customers * control['TARGET'].mean())
    n_prevented = int(n_expected_defaults * intervention_recovery_rate)
    capital = n_prevented * recovery_value
    cost = n_customers * cost_per_call
    net = capital - cost
    roi = (net / cost) * 100 if cost > 0 else 0

    rollout_results.append({
        'Stage': label,
        'Customers': n_customers,
        'Expected Defaults': n_expected_defaults,
        'Prevented': n_prevented,
        'Capital Recovered': capital,
        'Cost': cost,
        'Net Value': net,
        'ROI': roi
    })

    print(f"  {label}")
    print(f"    Customers called     : {n_customers:,}")
    print(f"    Expected defaults    : {n_expected_defaults:,}")
    print(f"    Defaults prevented   : {n_prevented:,}")
    print(f"    Capital recovered    : {capital:,.0f} CU")
    print(f"    Intervention cost    : {cost:,.0f} CU")
    print(f"    Net value            : {net:,.0f} CU")
    print(f"    ROI                  : {roi:,.1f}%")
    print()

print("=" * 60)

Controlled Rollout Model
At-risk population     : 35,952
Baseline default rate  : 9.78%
Observed reduction     : 2.86 pp
Recovery rate per call : 27.85%

  5% Canary
    Customers called     : 1,797
    Expected defaults    : 175
    Defaults prevented   : 48
    Capital recovered    : 16,608,168 CU
    Intervention cost    : 89,850 CU
    Net value            : 16,518,318 CU
    ROI                  : 18,384.3%

  20% Validation
    Customers called     : 7,190
    Expected defaults    : 702
    Defaults prevented   : 195
    Capital recovered    : 67,470,683 CU
    Intervention cost    : 359,500 CU
    Net value            : 67,111,183 CU
    ROI                  : 18,667.9%

  50% Scale
    Customers called     : 17,976
    Expected defaults    : 1,757
    Defaults prevented   : 489
    Capital recovered    : 169,195,714 CU
    Intervention cost    : 898,800 CU
    Net value            : 168,296,914 CU
    ROI                  : 18,724.6%

  100% Full Rollout
    Customers called   

### Controlled Rollout Finding

ROI is consistent across all four deployment stages, ranging from 18,384% to 18,725%. This
means the intervention does not lose its effectiveness at scale. Even the smallest canary
deployment of 1,797 customers recovers 16.5M CU net value.

The 5% canary stage serves as a safety check for Sleeping Dogs. If any customer segments
showed increased default rates after contact, this stage would catch it before scaling.
The 20% validation stage confirms the uplift holds with statistical significance at a
larger sample. By 50%, the financial case is proven and the remaining scale is operational.

At full rollout, the intervention prevents 978 defaults and recovers 336.6M CU against
a 1.8M CU cost. The recommendation is to proceed through all four stages sequentially,
with a stop-gate review after each one.

## Step 8 | Phase 3 Summary and Phase 4 Readiness